# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
#load detenv to work with variables in SRC DIR (env)
%load_ext dotenv  
%dotenv 

#import libraries and tools
import pandas as pd
import os
import sys
from glob import glob

sys.path.append(os.getenv('SRC_DIR'))

from utils.logger import get_logger
_logs = get_logger(__name__)


In [2]:
import dask.dataframe as dd #import from DASK library dataframe module

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [4]:
#to search files in directories
from glob import glob 

# the path to the data files
#search all the files .parquet including subdirectories
#read all the files with dask and use ticker like index
PRICE_DATA_ = os.getenv("PRICE_DATA") 
path_data_files = glob(os.path.join(PRICE_DATA_, '**/*.parquet'), 
                             recursive = True)  
dd_px = dd.read_parquet(path_data_files).set_index("ticker")

#to select 60 random files, and save in ft_glob (more faster than use all the data)
import random
random.seed(42)
ft_glob = random.sample(path_data_files, 60)

# to extract data from ft_glob and add to dt_list
dt_list = []
for s_file in ft_glob:
    _logs.info(f"Reading file: {s_file}")
    dt = dd.read_parquet(s_file)
    dt_list.append(dt)
price_data = dd.concat(dt_list, axis = 0, ignore_index = True) #To mix all files from dt_list  in one DataFrame price_data


2025-09-26 22:39:19,311, 770337610.py, 20, INFO, Reading file: ../../05_src/data/prices/ACB/ACB_2002/part.1.parquet
2025-09-26 22:39:19,313, 770337610.py, 20, INFO, Reading file: ../../05_src/data/prices/AMAT/AMAT_1990/part.0.parquet
2025-09-26 22:39:19,315, 770337610.py, 20, INFO, Reading file: ../../05_src/data/prices/AOSL/AOSL_2010/part.0.parquet
2025-09-26 22:39:19,317, 770337610.py, 20, INFO, Reading file: ../../05_src/data/prices/LYV/LYV_2011/part.0.parquet
2025-09-26 22:39:19,319, 770337610.py, 20, INFO, Reading file: ../../05_src/data/prices/CIK/CIK_2005/part.1.parquet
2025-09-26 22:39:19,321, 770337610.py, 20, INFO, Reading file: ../../05_src/data/prices/AIZ/AIZ_2018/part.0.parquet
2025-09-26 22:39:19,324, 770337610.py, 20, INFO, Reading file: ../../05_src/data/prices/EW/EW_2015/part.1.parquet
2025-09-26 22:39:19,328, 770337610.py, 20, INFO, Reading file: ../../05_src/data/prices/AMAT/AMAT_1993/part.1.parquet
2025-09-26 22:39:19,330, 770337610.py, 20, INFO, Reading file: ../..

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [5]:
#to group the dataframe by column ticker
# then crate new columns close_lag_1, Adj_Close_1, hi_lo_range
#.shift(1) moves data one row down let us compare the values
dd_shift = price_data.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(
        Close_lag_1 = x['Close'].shift(1),
        Adj_Close_1=x['Adj Close'].shift(1),
        hi_lo_range= lambda x: x['High']- x['Low']
        ),
) 

#  create a new column dd_shift and calc returns.
dd_feat = dd_shift.assign(
    returns = lambda df: df['Close']/df['Close_lag_1'] - 1
)

 

/var/folders/20/j844zscn2wndcgm54hxmfz480000gn/T/ipykernel_11197/1689759275.py:4: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_shift = price_data.groupby('ticker', group_keys=False).apply(


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [6]:
# Convert Dask data frame to Pandas dataframe
pandas_df_feat = dd_feat.compute()
pandas_df_feat['returns_ma_10'] = pandas_df_feat.groupby('ticker')['returns'].transform(lambda x: x.rolling(10).mean())
pandas_df_feat

,Date,Open,High,Low,Close,Adj Close,Volume,source,ticker,Year,Close_lag_1,Adj_Close_1,hi_lo_range,returns,returns_ma_10
131793,2002-07-03,7.43451,7.43451,7.43451,7.43451,7.434510,0.0,ACB.csv,ACB,2002,NaN,NaN,0.0000,NaN,NaN
131794,2002-07-05,7.43451,7.43451,7.43451,7.43451,7.434510,0.0,ACB.csv,ACB,2002,7.43451,7.434510,0.0000,0.000000,NaN
131795,2002-07-08,7.43451,7.43451,7.43451,7.43451,7.434510,0.0,ACB.csv,ACB,2002,7.43451,7.434510,0.0000,0.000000,NaN
131796,2002-07-09,7.35485,7.35485,7.35485,7.35485,7.354850,0.0,ACB.csv,ACB,2002,7.43451,7.434510,0.0000,-0.010715,NaN
131797,2002-07-10,7.32830,7.32830,7.32830,7.32830,7.328300,0.0,ACB.csv,ACB,2002,7.35485,7.354850,0.0000,-0.003610,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
196932,1997-12-24,15.25000,15.93750,15.12500,15.81250,14.482244,92600.0,ZEUS.csv,ZEUS,1997,15.25000,13.967071,0.8125,0.036885,0.019279
196933,1997-12-26,15.50000,16.37500,15.50000,16.37500,14.997422,88800.0,ZEUS.csv,ZEUS,1997,15.81250,14.482244,0.8750,0.035573,0.029440
196934,1997-12-29,16.37500,16.62500,15.37500,15.62500,14.310521,102300.0,ZEUS.csv,ZEUS,1997,16.37500,14.997422,1.2500,-0.045802,0.022839
196935,1997-12-30,15.62500,15.87500,15.00000,15.18750,13.909824,104600.0,ZEUS.csv,ZEUS,1997,15.62500,14.310521,0.8750,-0.028000,0.010138


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

1.  It  was not extrictly necessary, you can make the operations with Dask DataFrame, but Pandas DataFrame simplify the calculations, specially if you are working with small dataset or a little sample of the dataset.


2. It depends, because if you use only a little sample you can Use Pandas DataFrame, but if you need to use all the dataset and is a larger one is better Dask DataFrame.


## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.